# 03 — Static batching: the same 12 requests

**Question:** can the GPU finish 12 requests faster when it processes several together?

| Batch size | Sequential model calls | Total requests |
|---:|---:|---:|
| 1 | 12 | 12 |
| 3 | 4 | 12 |
| 6 | 2 | 12 |

Use a **fresh Colab T4 GPU session**. Run one section at a time. Keep the same GPU throughout.
We keep the previous model/checkpoint and use FP16. Each batch finishes before the next starts: **static batching**, not concurrent calls or continuous batching.
Start by predicting which row will have the highest throughput and which will use the most memory.

## 1. Install

Keep Colab's compatible PyTorch installation. If packages were already imported before an upgrade, restart the session before section 2.

In [1]:
%pip install -q "torch>=2.6,<3" "transformers==5.3.0"

## 2. Fix the experiment and check the GPU

Three measured rounds rotate the order: **1,3,6 → 3,6,1 → 6,1,3**. This reduces a consistent first/last-position advantage.
All settings and hardware details are saved; only a short hardware line is printed.

In [2]:
import csv, gc, json, platform, subprocess, sys, time, uuid  # Export data, inspect the runtime, and measure elapsed time.
from datetime import datetime, timezone  # Give each run a unique UTC identifier.
from importlib.metadata import version  # Detect imports left over from a package upgrade.
from pathlib import Path  # Keep each run in its own directory.
from statistics import mean, median  # Summarize repeated measurements.
from IPython.display import Markdown, display  # Display one readable comparison table.

for name in ['transformers', 'huggingface_hub']:
    loaded = sys.modules.get(name)  # Inspect any module already loaded in this kernel.
    if loaded is not None and loaded.__version__ != version(name.replace('_', '-')):
        raise RuntimeError('Restart the session after upgrading packages, then continue from section 2.')
import torch, transformers  # Execute the model and record the loaded library versions.
from transformers import AutoModelForCausalLM, AutoTokenizer, GenerationConfig  # Load matching weights and text rules.

MODEL = 'LiquidAI/LFM2.5-1.2B-Instruct'  # Reuse the model from the precision lesson.
REVISION = 'b9d816ab510f9a28dca37cb182e9da8b0226e13c'  # Pin the same exact checkpoint.
TOTAL_REQUESTS, BATCH_SIZES, ROUNDS = 12, [1, 3, 6], 3  # Change batching while keeping measured work fixed.
MAX_NEW_TOKENS, WARMUP_BATCHES = 64, 2  # Use the same output cap and warm every batch shape.
if not torch.cuda.is_available():  # Avoid silently benchmarking a CPU.
    raise RuntimeError('Select a GPU runtime in Colab, then reconnect and run from section 2.')
gc.collect(); torch.cuda.empty_cache()  # Release unused objects before checking for an old loaded model.
if torch.cuda.memory_allocated() > 0:  # Prevent a previous model from inflating this experiment's memory.
    raise RuntimeError('Start a fresh GPU session, then run this notebook in order.')
torch.manual_seed(42)  # Record a stable seed; greedy output may still vary numerically across batch sizes.
environment = {'python': platform.python_version(), 'torch': torch.__version__, 'transformers': transformers.__version__,
               'cuda_build': torch.version.cuda, 'gpu': torch.cuda.get_device_name(0),
               'gpu_total_gib': torch.cuda.get_device_properties(0).total_memory / 2**30}  # Record observed software and capacity.
environment['nvidia_smi'] = subprocess.check_output(['nvidia-smi', '--query-gpu=name,driver_version,memory.total,memory.free,power.limit', '--format=csv'], text=True).strip()  # Preserve driver, capacity and power limit.
print(f"GPU: {environment['gpu']} | FP16 | {environment['gpu_total_gib']:.1f} GiB | Transformers {transformers.__version__}")  # Keep setup output brief.

GPU: Tesla T4 | FP16 | 14.6 GiB | Transformers 5.3.0


## 3. Load the model once

Loading is outside the benchmark. `eval()` selects inference behavior; `inference_mode()` later avoids tracking gradients.
Greedy decoding chooses the highest-scoring next token. The 64-token setting is a maximum; EOS can end an answer earlier.

In [3]:
started = time.perf_counter()  # Measure setup separately from request execution.
tokenizer = AutoTokenizer.from_pretrained(MODEL, revision=REVISION, padding_side='left')  # Use decoder-friendly padding if needed later.
if tokenizer.pad_token_id is None:  # Give generation an explicit padding ID.
    tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(MODEL, revision=REVISION, dtype=torch.float16, attn_implementation='sdpa').to('cuda')  # Load one FP16 model on the GPU.
model.eval()  # Disable training-specific layer behavior.
torch.cuda.synchronize()  # Finish loading and device transfer before recording setup time.
load_seconds = time.perf_counter() - started  # Never include this value in request latency.
config = GenerationConfig(max_new_tokens=MAX_NEW_TOKENS, do_sample=False, num_beams=1, use_cache=True,
                          eos_token_id=model.generation_config.eos_token_id, pad_token_id=tokenizer.pad_token_id)  # Match generation behavior across batch sizes.
eos_ids = config.eos_token_id if isinstance(config.eos_token_id, list) else [config.eos_token_id]  # Support one or several valid stopping IDs.
print(f'Model loaded in {load_seconds:.1f}s; actual weight dtype: {next(model.parameters()).dtype}')  # Inspect setup without dumping the entire configuration.

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


config.json:   0%|          | 0.00/1.22k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/92.2k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/4.73M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/434 [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/5.49k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.34GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

Model loaded in 25.1s; actual weight dtype: torch.float16


## 4. Prepare the same 12 requests

Use **12 copies of one short prompt**, with IDs 1–12. This isolates batch size: equal input lengths, no input padding, same task and order.
These are synthetic requests, not 12 different questions or a quality evaluation. Tokenization and GPU input transfer happen before timing.
A tensor shape `(3, 66)` means **3 requests, each containing 66 input tokens**.

In [4]:
prompt = ('Service notes:\nThe service runs one language model on a GPU. Requests wait in a queue. '
          'Batching improves throughput but can increase waiting time. Longer inputs use more memory. '
          'The operator measures latency and errors.\n\nSummarize the service notes in exactly three short sentences.')  # Reuse the previous short workload.
rendered = tokenizer.apply_chat_template([{'role': 'user', 'content': prompt}], tokenize=False, add_generation_prompt=True)  # Add the model's chat markers.
prepared = {size: tokenizer([rendered] * size, padding=True, add_special_tokens=False, return_tensors='pt').to('cuda') for size in BATCH_SIZES}  # Build one reusable GPU input tensor per batch shape.
assert all(bool(inputs['attention_mask'].all()) for inputs in prepared.values())  # Confirm equal-length inputs need no padding.
input_tokens = prepared[1]['input_ids'].shape[1]  # Count actual tokens after chat formatting.
for size, inputs in prepared.items():
    print(f'Batch size {size}: input shape {tuple(inputs["input_ids"].shape)}, {TOTAL_REQUESTS // size} calls for 12 requests')  # Make the batching plan visible.

Batch size 1: input shape (1, 66), 12 calls for 12 requests
Batch size 3: input shape (3, 66), 4 calls for 12 requests
Batch size 6: input shape (6, 66), 2 calls for 12 requests


## 5. Create the result record

Results go into a unique folder in the **Colab machine**, not your PC. JSON preserves exact outputs and settings; CSV files will make comparison easy.

In [5]:
run_id = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ') + '-' + uuid.uuid4().hex[:8]  # Avoid overwriting earlier runs.
output_dir = Path('runs/static-batching') / run_id  # Keep this lesson separate from the precision experiment.
output_dir.mkdir(parents=True, exist_ok=False)  # Create the artifact directory.
requests, batches, workloads = [], [], []  # Preserve every response, model call, and completed 12-request workload.
payload = {'run_id': run_id, 'complete': False, 'environment': environment, 'model': MODEL, 'revision': REVISION,
           'dtype': str(next(model.parameters()).dtype), 'load_seconds': load_seconds, 'generation_config': config.to_dict(),
           'settings': {'total_requests': TOTAL_REQUESTS, 'batch_sizes': BATCH_SIZES, 'rounds': ROUNDS, 'warmup_batches': WARMUP_BATCHES, 'seed': 42},
           'prompt': prompt, 'rendered_prompt': rendered, 'input_token_ids': prepared[1]['input_ids'][0].tolist(),
           'timing_scope': 'All requests ready at workload start; inputs already on GPU. Total time includes generation and between-call Python/output-copy overhead, excludes loading, tokenization, text decoding, disk export and network.',
           'completion_policy': 'Each response is returned when its whole batch finishes; individual EOS arrival times are not observed.',
           'requests': requests, 'batches': batches, 'workloads': workloads}  # Keep full evidence rather than only medians.

def save_run():
    (output_dir / 'results.json').write_text(json.dumps(payload, indent=2, allow_nan=False) + '\n', encoding='utf-8')  # Save outside the timed workload.

save_run()  # Keep incomplete runs distinguishable if a later cell fails.
print('Results directory:', output_dir.resolve())  # Locate files before running the experiment.

Results directory: /content/runs/static-batching/20261122T234358Z-41c695f9


## 6. Define one workload measurement

The loop below makes one `generate()` call per batch. All requests are considered ready at time zero; the batches execute sequentially.

- **Batch generation time:** synchronized time for one model call.
- **Queue wait:** elapsed time before a request's batch starts.
- **Completion time:** elapsed time until the whole batch returns. This includes queue wait.
- **Total time:** completion time of the last batch, including small between-call overheads.
- **Throughput:** actual completed requests or generated tokens divided by total time.
- **Peak allocated GiB:** highest live tensor memory, including the model and prepared inputs.

This is an offline experiment, not an HTTP server. We do not measure TTFT/TPOT here. EOS counts as an output token; padding after EOS does not.

In [6]:
def run_workload(size, round_id, phase='measured'):
    count = TOTAL_REQUESTS if phase == 'measured' else size * WARMUP_BATCHES  # Warmups use two batches; measured cases always use 12 requests.
    inputs = prepared[size]  # Reuse prepared inputs while each generate call creates a fresh context cache.
    pending, call_times, error = [], [], None  # Defer text decoding and disk writes until timing ends.
    torch.cuda.synchronize(); torch.cuda.reset_peak_memory_stats()  # Exclude earlier GPU work and reset live-allocation peaks.
    origin = time.perf_counter()  # All requests in this workload are now considered ready.
    try:
        for offset in range(0, count, size):
            torch.cuda.synchronize()  # Ensure the previous batch has finished before starting this one.
            start = time.perf_counter()  # Record both batch service start and queue wait.
            with torch.inference_mode():  # Avoid gradient bookkeeping during inference.
                output = model.generate(**inputs, generation_config=config)  # Process exactly one static batch.
            torch.cuda.synchronize()  # Wait for actual GPU completion before stopping the batch timer.
            end = time.perf_counter()  # All requests in this batch are returned at this point.
            call_times.append(end - start)  # Retain generation-only time separately from total workload time.
            ids = output[:, inputs['input_ids'].shape[1]:].cpu().tolist()  # Remove input tokens and retain raw generated IDs.
            pending.append((offset, start - origin, end - origin, end - start, ids))  # Preserve each batch's timing and output.
            del output  # Avoid retaining GPU output tensors across batches.
    except Exception as exc:
        error = {'type': type(exc).__name__, 'message': str(exc)}  # Preserve failures without inventing timing for the failed call.
        raise
    finally:
        identity = {'phase': phase, 'round': round_id, 'batch_size': size}  # Identify warmups and measured repetitions separately.
        token_count = 0  # Sum actual generated tokens without post-EOS padding.
        for offset, wait, completion, service, outputs in pending:
            batches.append({**identity, 'batch_number': offset // size + 1, 'generation_seconds': service, 'queue_wait_seconds': wait, 'completion_seconds': completion})  # Preserve individual model-call data.
            for index, ids in enumerate(outputs):
                stop = next((position + 1 for position, token in enumerate(ids) if token in eos_ids), len(ids))  # Keep the first EOS and remove later output padding.
                tokens = ids[:stop]  # Count actual generated work for this response.
                token_count += len(tokens)  # Include terminal EOS but exclude later padding.
                requests.append({**identity, 'request_id': offset + index + 1, 'batch_number': offset // size + 1,
                                 'input_tokens': input_tokens, 'output_tokens': len(tokens), 'generated_token_ids': tokens,
                                 'response': tokenizer.decode(tokens, skip_special_tokens=True), 'finish_reason': 'eos' if tokens and tokens[-1] in eos_ids else 'length' if len(tokens) == MAX_NEW_TOKENS else 'other',
                                 'batch_generation_seconds': service, 'queue_wait_seconds': wait, 'completion_seconds': completion})  # Store every response after the timers stop.
        total = pending[-1][2] if pending and error is None else None  # Only a fully completed workload gets throughput metrics.
        workloads.append({**identity, 'status': 'error' if error else 'ok', 'error': error, 'completed_requests': sum(len(item[4]) for item in pending),
                          'model_calls': len(pending), 'total_seconds': total, 'generation_seconds': sum(call_times),
                          'requests_per_second': count / total if total else None, 'output_tokens': token_count, 'tokens_per_second': token_count / total if total else None,
                          'mean_batch_seconds': mean(call_times) if call_times else None, 'mean_completion_seconds': mean(item[2] for item in pending) if pending else None,
                          'peak_allocated_gib': torch.cuda.max_memory_allocated() / 2**30, 'peak_reserved_gib': torch.cuda.max_memory_reserved() / 2**30})  # Reserved memory may retain cached blocks from previous shapes.
        save_run()  # Checkpoint completed responses and any error outside all timing intervals.

## 7. Run the comparison

First warm each batch size with two calls. Then execute **12 requests per configuration per round**, for three rounds.
This gives **108 measured responses**, plus 20 warmup responses. Only measured workloads enter the comparison table.

In [7]:
if workloads:  # Prevent an accidental rerun from mixing duplicate measurements into this run.
    raise RuntimeError('Rerun section 5 to create a new run, then execute section 7 again.')
for size in BATCH_SIZES:
    run_workload(size, 0, phase='warmup')  # Warm every batch shape and preserve its responses separately.
for index in range(ROUNDS):
    order = BATCH_SIZES[index % len(BATCH_SIZES):] + BATCH_SIZES[:index % len(BATCH_SIZES)]  # Rotate which batch size runs first.
    for size in order:
        run_workload(size, index + 1)  # Complete the same 12 requests for each configuration.
    print(f'Round {index + 1}/{ROUNDS} complete; batch-size order: {order}')  # Print progress only between timed workloads.
payload['complete'] = True  # Mark complete only when every requested workload succeeds.
save_run()  # Preserve the completed run before displaying or exporting it.

Round 1/3 complete; batch-size order: [1, 3, 6]
Round 2/3 complete; batch-size order: [3, 6, 1]
Round 3/3 complete; batch-size order: [6, 1, 3]


## 8. Read just this table first

Each row contains medians across three measured rounds. **Lower total time and higher throughput are better; memory is a resource cost.**
`mean_completion_seconds` in the CSV includes waiting for earlier batches. `total_seconds / 12` would be amortized time, not an individual request's latency.
Model calls can produce different output lengths numerically: compare counts, stop reasons and answers alongside speed.

In [8]:
measured = [row for row in requests if row['phase'] == 'measured']  # Exclude all warmup responses.
references = {row['request_id']: row['generated_token_ids'] for row in measured if row['batch_size'] == 1 and row['round'] == 1}  # Use real sequential outputs as the comparison reference.
for row in requests:
    row['matches_sequential_ids'] = row['generated_token_ids'] == references.get(row['request_id']) if row['phase'] == 'measured' else None  # Equality is a consistency check, not a quality score.
summaries = []  # Keep rounded display summaries separate from exact measurements.
metrics = ['total_seconds', 'generation_seconds', 'requests_per_second', 'tokens_per_second', 'mean_batch_seconds', 'mean_completion_seconds', 'peak_allocated_gib', 'peak_reserved_gib']  # Preserve important metrics without displaying every raw value.
for size in BATCH_SIZES:
    cases = [row for row in workloads if row['phase'] == 'measured' and row['batch_size'] == size and row['status'] == 'ok']  # Compare complete workloads only.
    answers = [row for row in measured if row['batch_size'] == size]  # Check every measured response for this batch size.
    summary = {'batch_size': size, 'calls_per_12_requests': TOTAL_REQUESTS // size, 'measured_rounds': len(cases)}  # State sample counts explicitly.
    summary.update({f'median_{key}': median(row[key] for row in cases) for key in metrics})  # Calculate medians from the raw workload records.
    summary.update({'min_output_tokens': min(row['output_tokens'] for row in answers), 'max_output_tokens': max(row['output_tokens'] for row in answers), 'all_match_sequential': all(row['matches_sequential_ids'] for row in answers)})  # Keep output changes visible.
    summaries.append(summary)  # Save one compact comparison row per batch size.
table = '| Batch size | Calls / 12 | Total s | Requests/s | Tokens/s | Peak allocated GiB | Outputs match? |\n|---:|---:|---:|---:|---:|---:|:---:|\n'  # Label units in the visible table.
for row in summaries:
    table += f"| {row['batch_size']} | {row['calls_per_12_requests']} | {row['median_total_seconds']:.3f} | {row['median_requests_per_second']:.2f} | {row['median_tokens_per_second']:.1f} | {row['median_peak_allocated_gib']:.3f} | {row['all_match_sequential']} |\n"  # Print three rows instead of full JSON.
display(Markdown(table))  # Show the main comparison.
print('One actual response:', measured[0]['response'])  # Inspect an answer without printing all 108 responses.
payload['summaries'] = summaries  # Retain displayed summaries with exact raw evidence.
save_run()  # Preserve response-equality checks as well as measurements.

| Batch size | Calls / 12 | Total s | Requests/s | Tokens/s | Peak allocated GiB | Outputs match? |
|---:|---:|---:|---:|---:|---:|:---:|
| 1 | 12 | 9.347 | 1.28 | 52.6 | 2.194 | True |
| 3 | 4 | 3.274 | 3.67 | 150.3 | 2.205 | True |
| 6 | 2 | 1.677 | 7.15 | 293.3 | 2.219 | True |


One actual response: The service uses a single language model on a GPU, with requests processed in a queue. Batching helps speed up responses but may extend wait times. Longer inputs require more memory and affect performance.


## 9. Save and download

- `summary.csv`: three comparison rows, including output lengths, batch latency and mean request completion time.
- `workloads.csv` / `batches.csv`: individual measured/warmup workloads and calls.
- `requests.csv`: every response, actual output length, waiting/completion times and equality check.
- `results.json`: everything above plus full token IDs, prompts, hardware and generation settings.

Open the CSVs only when needed. Download the ZIP before Colab storage disappears. After reviewing a real GPU run, preserve selected results under the repository's `results/` folder; `runs/` is intentionally ignored by Git.

In [9]:
def write_csv(name, rows, omit=()):
    columns = [key for key in rows[0] if key not in omit]  # Keep large token arrays in JSON rather than CSV.
    with (output_dir / name).open('w', newline='', encoding='utf-8') as handle:
        writer = csv.DictWriter(handle, fieldnames=columns, extrasaction='ignore')  # Quote responses correctly when they contain commas or newlines.
        writer.writeheader(); writer.writerows(rows)  # Write one row per actual measurement or response.

write_csv('summary.csv', summaries)  # Save the compact comparison.
write_csv('workloads.csv', workloads, omit=('error',))  # Save individual full-workload measurements.
write_csv('batches.csv', batches)  # Save batch generation, wait and completion times.
write_csv('requests.csv', requests, omit=('generated_token_ids',))  # Preserve every response in a readable table.
print('Saved:', ', '.join(path.name for path in sorted(output_dir.iterdir())))  # Show filenames without dumping their contents.
print('Directory:', output_dir.resolve())  # Locate files in Colab's Files panel.

Saved: batches.csv, requests.csv, results.json, summary.csv, workloads.csv
Directory: /content/runs/static-batching/20261122T234358Z-41c695f9


In [ ]:
# Optional in Colab: uncomment these lines to download all artifacts together.
# import shutil
# from google.colab import files
# archive = shutil.make_archive(str(output_dir), 'zip', root_dir=output_dir)
# files.download(archive)

## 10. Explain three things before the next experiment

1. Did larger batches finish all 12 requests sooner? Compare total time and aggregate throughput.
2. Did batch latency or GPU memory rise? Inspect `summary.csv` for batch latency and request completion time.
3. Did actual output lengths, stop reasons or responses change? Identical outputs do not prove correctness.

**Limits:** one repeated prompt, one GPU, three rounds. No p95/p99, network, continuous-batching or token-level-latency claims. Reserved memory includes allocator blocks retained from earlier shapes; it is not extra live tensor memory. The next experiment can introduce mixed-length prompts and padding.

References: [Transformers batching and left padding](https://huggingface.co/docs/transformers/v5.3.0/en/llm_tutorial), [generation settings](https://huggingface.co/docs/transformers/v5.3.0/en/main_classes/text_generation), [PyTorch CUDA timing and memory](https://docs.pytorch.org/docs/stable/notes/cuda.html).